In [ ]:
!pip install ultralytics roboflow -q

from roboflow import Roboflow
rf = Roboflow(api_key="uzSIgudwyOp9TryMxumT")

# Your dataset (from your fork)
your_proj = rf.workspace("ashmita-kamath131207-gmail-com").project("ict320-y12-horseracing-dataset-b7ipi-ofr6d")
your_ds = your_proj.version(1).download("yolov11", location="/kaggle/working/your_data")

# Their dataset (horse + jockey)
their_proj = rf.workspace("horse-detection-yolov12").project("horse-detection-yolov12-v2")
their_ds = their_proj.version(2).download("yolov11", location="/kaggle/working/their_data")

print("Both downloaded.")

In [ ]:
import yaml
for name, path in [("YOURS", "/kaggle/working/your_data/data.yaml"), ("THEIRS", "/kaggle/working/their_data/data.yaml")]:
    with open(path) as f:
        d = yaml.safe_load(f)
    print(name, "-> classes:", d.get("names"))

In [ ]:
import os, shutil, glob

MERGED = "/kaggle/working/merged"
for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED}/{split}/labels", exist_ok=True)

def copy_split(src_root, split, drop_jockey):
    img_dir = f"{src_root}/{split}/images"
    lbl_dir = f"{src_root}/{split}/labels"
    if not os.path.isdir(img_dir):
        return 0
    count = 0
    for img in glob.glob(f"{img_dir}/*"):
        base = os.path.splitext(os.path.basename(img))[0]
        lbl = f"{lbl_dir}/{base}.txt"
        shutil.copy(img, f"{MERGED}/{split}/images/")
        new_lines = []
        if os.path.exists(lbl):
            with open(lbl) as f:
                for line in f:
                    parts = line.split()
                    if not parts:
                        continue
                    cls = int(parts[0])
                    if drop_jockey:
                        if cls == 1:
                            continue
                        parts[0] = "0"
                    else:
                        parts[0] = "0"
                    new_lines.append(" ".join(parts))
        with open(f"{MERGED}/{split}/labels/{base}.txt", "w") as f:
            f.write("\n".join(new_lines))
        count += 1
    return count

for split in ["train", "valid", "test"]:
    y = copy_split("/kaggle/working/your_data", split, drop_jockey=False)
    t = copy_split("/kaggle/working/their_data", split, drop_jockey=True)
    print(f"{split}: yours={y}, theirs={t}")

with open(f"{MERGED}/data.yaml", "w") as f:
    f.write("train: /kaggle/working/merged/train/images\n")
    f.write("val: /kaggle/working/merged/valid/images\n")
    f.write("test: /kaggle/working/merged/test/images\n")
    f.write("nc: 1\n")
    f.write("names: ['racing-horse']\n")

print("Merge done.")

In [ ]:
import glob
bad = 0
for lbl in glob.glob("/kaggle/working/merged/*/labels/*.txt"):
    with open(lbl) as f:
        for line in f:
            if line.strip() and line.split()[0] != "0":
                bad += 1
print("Non-zero class lines found:", bad, "(should be 0)")

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11m.pt")
model.train(
    data="/kaggle/working/merged/data.yaml",
    epochs=128, imgsz=960, batch=16,
    device=0, patience=50, save_period=10,
    project="horse_racing", name="combined_960",
)
model.val()

In [ ]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/detect/train/weights/last.pt')
model.train(resume=True)

In [ ]:
!pip install ultralytics -q

# Check if weights survived
!find /kaggle/working -name "last.pt"

In [ ]:
!find /kaggle/working -name "*.pt"

In [ ]:
!ls /kaggle/working/merged/
!cat /kaggle/working/merged/data.yaml